In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from hakken_agents.config import EmbedderConfig
from hakken_agents.db.config import PostgresDBConfig
from hakken_agents.vector_db.config import VectorDBConfig, VectorDBTableConfig
from hakken_agents.vector_db.engine import VectorDBEngine

db_config = PostgresDBConfig()

embeddings_config = EmbedderConfig(
    api_key=os.getenv("OPENAI_API_KEY"), embedding_model="text-embedding-3-small"
)

table_name = "domains_vectors"
table_config = VectorDBTableConfig(
    name=table_name,
    schema_name="public",
    content_column="content",
    embedding_column="embedding",
    metadata_columns=["level_1", "level_2", "level_3", "level_4"],
)
vector_db_config = VectorDBConfig(
    db=db_config,
    embedder=embeddings_config,
    table=table_config,
)

print(db_config)

user='postgres' password='postgres' host='localhost' port=5432 database='hakken_agents'


In [4]:
vector_db = await VectorDBEngine.create_from_config(vector_db_config)

In [11]:
count = await vector_db.acount_documents(table_name=table_name, schema_name="public")
print(count)
docs = await vector_db.alist_documents(
    table_name=table_name, schema_name="public", id_column="langchain_id"
)

10


In [12]:
from langchain_core.documents import Document

initial_docs = [
    Document(page_content="disease", metadata={"level_1": "disease"}),
    Document(
        page_content="disease/respiratory_disease",
        metadata={"level_1": "disease", "level_2": "respiratory_disease"},
    ),
    Document(page_content="disease/cancer", metadata={"level_1": "disease", "level_2": "cancer"}),
    Document(
        page_content="disease/cancer/breast_cancer",
        metadata={"level_1": "disease", "level_2": "cancer", "level_3": "breast_cancer"},
    ),
    Document(page_content="gene", metadata={"level_1": "gene"}),
    Document(page_content="gene/human_gene", metadata={"level_1": "gene", "level_2": "human_gene"}),
    Document(
        page_content="gene/animal_gene", metadata={"level_1": "gene", "level_2": "animal_gene"}
    ),
    Document(page_content="chemical", metadata={"level_1": "chemical"}),
    Document(page_content="chemical/drug", metadata={"level_1": "chemical", "level_2": "drug"}),
    Document(
        page_content="chemical/drug/antibacterial_drug",
        metadata={"level_1": "chemical", "level_2": "drug", "level_3": "antibacterial_drug"},
    ),
]

In [13]:
if count < len(initial_docs):
    await vector_db.aadd_documents(initial_docs)

In [ ]:
new_domain = "Chemical/Drug/viral_drug"

new_domain_parts = new_domain.split("/")

docs = await vector_db._store.asimilarity_search(new_domain, k=10_000)
for doc in docs:
    print(doc)


docs = await vector_db._store.asimilarity_search(new_domain_parts[0], k=10_000)
for doc in docs:
    print(doc)

page_content='chemical/drug' metadata={'level_1': 'chemical', 'level_2': 'drug'}
page_content='chemical/drug/antibacterial_drug' metadata={'level_1': 'chemical', 'level_2': 'drug', 'level_3': 'antibacterial_drug'}
page_content='chemical' metadata={'level_1': 'chemical'}
page_content='disease/cancer' metadata={'level_1': 'disease', 'level_2': 'cancer'}
page_content='disease/respiratory_disease' metadata={'level_1': 'disease', 'level_2': 'respiratory_disease'}
page_content='gene/human_gene' metadata={'level_1': 'gene', 'level_2': 'human_gene'}
page_content='disease' metadata={'level_1': 'disease'}
page_content='disease/cancer/breast_cancer' metadata={'level_1': 'disease', 'level_2': 'cancer', 'level_3': 'breast_cancer'}
page_content='gene/animal_gene' metadata={'level_1': 'gene', 'level_2': 'animal_gene'}
page_content='gene' metadata={'level_1': 'gene'}
page_content='chemical' metadata={'level_1': 'chemical'}
page_content='chemical/drug' metadata={'level_1': 'chemical', 'level_2': 'drug'